In [1]:
pip install pandas sqlalchemy psycopg2-binary pyodbc

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.7 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 16.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
PG_HOST= "aws-0-us-west-2.pooler.supabase.com"
PG_PORT= "6543"
PG_NAME= "postgres"
PG_USER= "postgres.pewsvvqdxlmuyerlwudt"
PG_PASS= "casadesucos@"

In [5]:
import pandas as pd
import urllib.parse
import os
from sqlalchemy import create_engine, text

# ==========================================
# CONFIGURAÇÃO DOS BANCOS DE ORIGEM (SQL SERVER)
# ==========================================
SQLSERVER_HOST = "localhost"

BANCOS_ORIGEM = [
    {
        "database": "sige_dados",
        "origem_banco": "GRUPO_BARRA",
        "empresas_permitidas": [
            "CASA DE SUCO - BARRA DO CORDA",
            "EMPORIO MIX"
        ]
    },
    {
        "database": "sige_itz",
        "origem_banco": "GRUPO_ITZ",
        "empresas_permitidas": [
            "PDV ITZ 01",
            "PDV ITZ 02",
            "PDV ITZ 03",
            "PDV ITZ 04"
        ]
    }
]

TABELA_ORIGEM = "pedidos"
TABELA_DESTINO = "pedidos"
CHUNK_SIZE = 5000

# ==========================================
# CONFIGURAÇÃO DO POSTGRESQL (SUPABASE / POOLER)
# ==========================================
PG_USER = "postgres.pewsvvqdxlmuyerlwudt"
PG_PASS = urllib.parse.quote_plus("casadesucos@")
PG_HOST = "aws-0-us-west-2.pooler.supabase.com"
PG_PORT = "6543"
PG_DB = "postgres"

print("PG_HOST:", PG_HOST)
print("PG_PORT:", PG_PORT)
print("PG_DB:", PG_DB)
print("PG_USER:", PG_USER)

engine_postgres = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    connect_args={"sslmode": "require"}
)

# ==========================================
# FUNÇÕES AUXILIARES
# ==========================================
def criar_engine_sqlserver(database: str):
    params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={SQLSERVER_HOST};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
    )
    return create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


def criar_tabela_se_nao_existe():
    query = f'''
    CREATE TABLE IF NOT EXISTS "{TABELA_DESTINO}" (
        "ID_UNICO" TEXT PRIMARY KEY
    );
    '''
    with engine_postgres.begin() as conn:
        conn.execute(text(query))


def garantir_colunas_existentes(df, tabela):
    if df.empty:
        return

    query_colunas_existentes = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = :tabela
    """)

    with engine_postgres.begin() as conn:
        resultado = conn.execute(query_colunas_existentes, {"tabela": tabela})
        colunas_existentes = {row[0] for row in resultado.fetchall()}

        for coluna in df.columns:
            if coluna not in colunas_existentes:
                conn.execute(text(f'ALTER TABLE "{tabela}" ADD COLUMN "{coluna}" TEXT'))
                print(f"🧱 Coluna criada: {coluna}")


def ajustar_pk_id_unico(tabela):
    with engine_postgres.begin() as conn:
        conn.execute(text(f'ALTER TABLE "{tabela}" ADD COLUMN IF NOT EXISTS "ID_UNICO" TEXT'))

        pk_query = text("""
            SELECT con.conname
            FROM pg_constraint con
            JOIN pg_class rel ON rel.oid = con.conrelid
            WHERE rel.relname = :tabela
              AND con.contype = 'p'
        """)

        resultado = conn.execute(pk_query, {"tabela": tabela}).fetchone()

        if resultado:
            nome_pk = resultado[0]
            try:
                conn.execute(text(f'ALTER TABLE "{tabela}" DROP CONSTRAINT "{nome_pk}"'))
                print(f"🔧 PK antiga removida: {nome_pk}")
            except Exception as e:
                print(f"⚠️ Não foi possível remover PK antiga ({nome_pk}): {e}")

        try:
            conn.execute(text(f'ALTER TABLE "{tabela}" ADD PRIMARY KEY ("ID_UNICO")'))
            print('✅ PK definida em "ID_UNICO"')
        except Exception as e:
            print(f"ℹ️ PK em ID_UNICO já existe ou não precisou ser alterada: {e}")


def preparar_dataframe(df: pd.DataFrame, origem_banco: str, empresas_permitidas: list) -> pd.DataFrame:
    df = df.loc[:, ~df.columns.duplicated()].copy()

    if "ID" not in df.columns:
        raise ValueError("A tabela de origem não possui a coluna 'ID'.")

    if "Empresa" not in df.columns:
        raise ValueError("A tabela de origem não possui a coluna 'Empresa'.")

    df["Empresa"] = df["Empresa"].astype(str).str.strip()

    empresas_permitidas_normalizadas = {e.strip() for e in empresas_permitidas}
    df = df[df["Empresa"].isin(empresas_permitidas_normalizadas)].copy()

    df["OrigemBanco"] = origem_banco
    df["ID_UNICO"] = (
        df["OrigemBanco"].astype(str).str.strip() + "_" +
        df["Empresa"].astype(str).str.strip() + "_" +
        df["ID"].astype(str).str.strip()
    )

    for col in df.columns:
        df[col] = df[col].astype(str).where(df[col].notna(), None)

    return df


def upsert_postgres(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return

    df = df.loc[:, ~df.columns.duplicated()].copy()

    criar_tabela_se_nao_existe()
    garantir_colunas_existentes(df, tabela)
    ajustar_pk_id_unico(tabela)

    tabela_temp = f"{tabela}_temp"

    df.to_sql(tabela_temp, engine_postgres, if_exists="replace", index=False)

    colunas = list(df.columns)

    insert_cols = ', '.join(f'"{c}"' for c in colunas)
    select_cols = ', '.join(f'"{c}"' for c in colunas)
    update_set = ', '.join(
        f'"{c}" = EXCLUDED."{c}"' for c in colunas if c != "ID_UNICO"
    )

    query = f"""
    INSERT INTO "{tabela}" ({insert_cols})
    SELECT {select_cols} FROM "{tabela_temp}"
    ON CONFLICT ("ID_UNICO") DO UPDATE SET
    {update_set};
    """

    with engine_postgres.begin() as conn:
        conn.execute(text(query))
        conn.execute(text(f'DROP TABLE IF EXISTS "{tabela_temp}"'))

    print(f"✅ Lote gravado/upsert: {len(df)} registros")


def migrar_banco_sqlserver(database: str, origem_banco: str, empresas_permitidas: list):
    print(f"\n🚀 Iniciando migração | Banco: {database} | Origem: {origem_banco}")

    engine_sqlserver = criar_engine_sqlserver(database)
    query = f"SELECT * FROM {TABELA_ORIGEM}"

    total_lido = 0
    total_filtrado = 0

    for i, chunk in enumerate(pd.read_sql(query, engine_sqlserver, chunksize=CHUNK_SIZE), start=1):
        print(f"📦 Lote {i} | Banco {database} | {len(chunk)} registros lidos")
        total_lido += len(chunk)

        chunk = preparar_dataframe(chunk, origem_banco, empresas_permitidas)

        if chunk.empty:
            print(f"⚠️ Lote {i} sem registros após filtro de empresas")
            continue

        total_filtrado += len(chunk)
        upsert_postgres(chunk, TABELA_DESTINO)

    print(f"🏁 Banco finalizado: {database}")
    print(f"   Total lido: {total_lido}")
    print(f"   Total válido após filtro: {total_filtrado}")


def migrar_historico_completo():
    print("============================================")
    print("🚀 MIGRAÇÃO COMPLETA SQL SERVER -> POSTGRES")
    print("============================================")

    for idx, origem in enumerate(BANCOS_ORIGEM, start=1):
        print(f"\n=== Banco {idx} de {len(BANCOS_ORIGEM)} ===")
        migrar_banco_sqlserver(
            database=origem["database"],
            origem_banco=origem["origem_banco"],
            empresas_permitidas=origem["empresas_permitidas"]
        )

    print("\n✅ Migração completa finalizada com sucesso!")


if __name__ == "__main__":
    migrar_historico_completo()

PG_HOST: aws-0-us-west-2.pooler.supabase.com
PG_PORT: 6543
PG_DB: postgres
PG_USER: postgres.pewsvvqdxlmuyerlwudt
🚀 MIGRAÇÃO COMPLETA SQL SERVER -> POSTGRES

=== Banco 1 de 2 ===

🚀 Iniciando migração | Banco: sige_dados | Origem: GRUPO_BARRA
📦 Lote 1 | Banco sige_dados | 5000 registros lidos
🧱 Coluna criada: OrigemBanco
🧱 Coluna criada: ID_UNICO
🔧 PK antiga removida: pedidos_pkey
ℹ️ PK em ID_UNICO já existe ou não precisou ser alterada: (psycopg2.errors.NotNullViolation) column "ID_UNICO" of relation "pedidos" contains null values

[SQL: ALTER TABLE "pedidos" ADD PRIMARY KEY ("ID_UNICO")]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


ProgrammingError: (psycopg2.errors.InvalidColumnReference) there is no unique or exclusion constraint matching the ON CONFLICT specification

[SQL: 
    INSERT INTO "pedidos" ("ID", "AndroidVendaID", "Codigo", "OrigemVenda", "DepositoID", "Tabela", "TabelaID", "Deposito", "StatusSistema", "Status", "Categoria", "Validade", "Empresa", "EmpresaID", "ClienteID", "Cliente", "PessoaID", "ClienteCNPJ", "ClienteEmail", "Vendedor", "VendedorID", "PlanoDeConta", "FormaPagamento", "FormaPagamentoID", "NumeroParcelas", "FreteMeioEnvio", "Transportadora", "FreteFormaEnvio", "DataEnvio", "PrevisaoEntrega", "DataPostagem", "Enviado", "ValorFrete", "FreteContaEmitente", "CodigoRastreio", "EnderecoOpcional", "ValorSeguro", "Descricao", "OutrasDespesas", "TransacaoCartao", "ValorFinal", "Finalizado", "Lancado", "Municipio", "CodigoMunicipio", "Pais", "CEP", "UF", "UFCodigo", "Bairro", "Logradouro", "LogradouroNumero", "LogradouroComplemento", "GruposProdutos", "Items", "Banco", "Data", "Pagamentos", "L

In [7]:
import pandas as pd
import urllib.parse
import os
from sqlalchemy import create_engine, text

# ==========================================
# CONFIGURAÇÃO DOS BANCOS DE ORIGEM (SQL SERVER)
# ==========================================
SQLSERVER_HOST = "localhost"

BANCOS_ORIGEM = [
    {
        "database": "sige_dados",
        "origem_banco": "GRUPO_BARRA",
        "empresas_permitidas": [
            "CASA DE SUCO - BARRA DO CORDA",
            "EMPORIO MIX"
        ]
    },
    {
        "database": "sige_itz",
        "origem_banco": "GRUPO_ITZ",
        "empresas_permitidas": [
            "PDV ITZ 01",
            "PDV ITZ 02",
            "PDV ITZ 03",
            "PDV ITZ 04"
        ]
    }
]

TABELA_ORIGEM = "pedidos"
TABELA_DESTINO = "pedidos"
CHUNK_SIZE = 5000

# ==========================================
# CONFIGURAÇÃO DO POSTGRESQL
# ==========================================
PG_USER = "postgres.pewsvvqdxlmuyerlwudt"
PG_PASS = urllib.parse.quote_plus("casadesucos@")
PG_HOST = "aws-0-us-west-2.pooler.supabase.com"
PG_PORT = "6543"
PG_DB = "postgres"

print("PG_HOST:", PG_HOST)
print("PG_PORT:", PG_PORT)
print("PG_DB:", PG_DB)
print("PG_USER:", PG_USER)

engine_postgres = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    connect_args={"sslmode": "require"}
)

# ==========================================
# FUNÇÕES AUXILIARES
# ==========================================
def criar_engine_sqlserver(database: str):
    params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={SQLSERVER_HOST};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
    )
    return create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


def criar_tabela_se_nao_existe():
    query = f'''
    CREATE TABLE IF NOT EXISTS "{TABELA_DESTINO}" (
        "ID_UNICO" TEXT
    );
    '''
    with engine_postgres.begin() as conn:
        conn.execute(text(query))


def garantir_colunas_existentes(df, tabela):
    if df.empty:
        return

    query_colunas_existentes = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = :tabela
    """)

    with engine_postgres.begin() as conn:
        resultado = conn.execute(query_colunas_existentes, {"tabela": tabela})
        colunas_existentes = {row[0] for row in resultado.fetchall()}

        for coluna in df.columns:
            if coluna not in colunas_existentes:
                conn.execute(text(f'ALTER TABLE "{tabela}" ADD COLUMN "{coluna}" TEXT'))
                print(f"🧱 Coluna criada: {coluna}")


def ajustar_constraint_id_unico(tabela):
    with engine_postgres.begin() as conn:
        conn.execute(text(f'ALTER TABLE "{tabela}" ADD COLUMN IF NOT EXISTS "ID_UNICO" TEXT'))

        conn.execute(text(f'''
            UPDATE "{tabela}"
            SET "ID_UNICO" = COALESCE("OrigemBanco",'SEM_ORIGEM') || '_' ||
                             COALESCE("Empresa",'SEM_EMPRESA') || '_' ||
                             COALESCE("ID",'SEM_ID')
            WHERE "ID_UNICO" IS NULL
        '''))

        # Remove duplicados por ID_UNICO
        conn.execute(text(f'''
            DELETE FROM "{tabela}" a
            USING "{tabela}" b
            WHERE a.ctid < b.ctid
              AND a."ID_UNICO" = b."ID_UNICO"
        '''))

        # Remove PK antiga em ID, se existir
        conn.execute(text(f'''
            DO $$
            DECLARE
                pk_name text;
            BEGIN
                SELECT con.conname
                INTO pk_name
                FROM pg_constraint con
                JOIN pg_class rel ON rel.oid = con.conrelid
                WHERE rel.relname = '{tabela}'
                  AND con.contype = 'p';

                IF pk_name IS NOT NULL THEN
                    EXECUTE format('ALTER TABLE "{tabela}" DROP CONSTRAINT %I', pk_name);
                END IF;
            END $$;
        '''))

        # Remove UNIQUE antiga em ID_UNICO, se existir
        conn.execute(text(f'''
            ALTER TABLE "{tabela}"
            DROP CONSTRAINT IF EXISTS "{tabela}_id_unico_key"
        '''))

        # Cria UNIQUE nova em ID_UNICO
        conn.execute(text(f'''
            ALTER TABLE "{tabela}"
            ADD CONSTRAINT "{tabela}_id_unico_key" UNIQUE ("ID_UNICO")
        '''))

        print('✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida')


def preparar_dataframe(df: pd.DataFrame, origem_banco: str, empresas_permitidas: list) -> pd.DataFrame:
    df = df.loc[:, ~df.columns.duplicated()].copy()

    if "ID" not in df.columns:
        raise ValueError("A tabela de origem não possui a coluna 'ID'.")

    if "Empresa" not in df.columns:
        raise ValueError("A tabela de origem não possui a coluna 'Empresa'.")

    df["Empresa"] = df["Empresa"].astype(str).str.strip()

    empresas_permitidas_normalizadas = {e.strip() for e in empresas_permitidas}
    df = df[df["Empresa"].isin(empresas_permitidas_normalizadas)].copy()

    df["OrigemBanco"] = origem_banco
    df["ID_UNICO"] = (
        df["OrigemBanco"].astype(str).str.strip() + "_" +
        df["Empresa"].astype(str).str.strip() + "_" +
        df["ID"].astype(str).str.strip()
    )

    for col in df.columns:
        df[col] = df[col].astype(str).where(df[col].notna(), None)

    return df


def upsert_postgres(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return

    df = df.loc[:, ~df.columns.duplicated()].copy()

    criar_tabela_se_nao_existe()
    garantir_colunas_existentes(df, tabela)
    ajustar_constraint_id_unico(tabela)

    tabela_temp = f"{tabela}_temp"

    df.to_sql(tabela_temp, engine_postgres, if_exists="replace", index=False)

    colunas = list(df.columns)

    insert_cols = ', '.join(f'"{c}"' for c in colunas)
    select_cols = ', '.join(f'"{c}"' for c in colunas)
    update_set = ', '.join(
        f'"{c}" = EXCLUDED."{c}"' for c in colunas if c != "ID_UNICO"
    )

    query = f"""
    INSERT INTO "{tabela}" ({insert_cols})
    SELECT {select_cols} FROM "{tabela_temp}"
    ON CONFLICT ("ID_UNICO") DO UPDATE SET
    {update_set};
    """

    with engine_postgres.begin() as conn:
        conn.execute(text(query))
        conn.execute(text(f'DROP TABLE IF EXISTS "{tabela_temp}"'))

    print(f"✅ Lote gravado/upsert: {len(df)} registros")


def migrar_banco_sqlserver(database: str, origem_banco: str, empresas_permitidas: list):
    print(f"\n🚀 Iniciando migração | Banco: {database} | Origem: {origem_banco}")

    engine_sqlserver = criar_engine_sqlserver(database)
    query = f"SELECT * FROM {TABELA_ORIGEM}"

    total_lido = 0
    total_filtrado = 0

    for i, chunk in enumerate(pd.read_sql(query, engine_sqlserver, chunksize=CHUNK_SIZE), start=1):
        print(f"📦 Lote {i} | Banco {database} | {len(chunk)} registros lidos")
        total_lido += len(chunk)

        chunk = preparar_dataframe(chunk, origem_banco, empresas_permitidas)

        if chunk.empty:
            print(f"⚠️ Lote {i} sem registros após filtro de empresas")
            continue

        total_filtrado += len(chunk)
        upsert_postgres(chunk, TABELA_DESTINO)

    print(f"🏁 Banco finalizado: {database}")
    print(f"   Total lido: {total_lido}")
    print(f"   Total válido após filtro: {total_filtrado}")


def migrar_historico_completo():
    print("============================================")
    print("🚀 MIGRAÇÃO COMPLETA SQL SERVER -> POSTGRES")
    print("============================================")

    for idx, origem in enumerate(BANCOS_ORIGEM, start=1):
        print(f"\n=== Banco {idx} de {len(BANCOS_ORIGEM)} ===")
        migrar_banco_sqlserver(
            database=origem["database"],
            origem_banco=origem["origem_banco"],
            empresas_permitidas=origem["empresas_permitidas"]
        )

    print("\n✅ Migração completa finalizada com sucesso!")


if __name__ == "__main__":
    migrar_historico_completo()

PG_HOST: aws-0-us-west-2.pooler.supabase.com
PG_PORT: 6543
PG_DB: postgres
PG_USER: postgres.pewsvvqdxlmuyerlwudt
🚀 MIGRAÇÃO COMPLETA SQL SERVER -> POSTGRES

=== Banco 1 de 2 ===

🚀 Iniciando migração | Banco: sige_dados | Origem: GRUPO_BARRA
📦 Lote 1 | Banco sige_dados | 5000 registros lidos
✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida
✅ Lote gravado/upsert: 5000 registros
📦 Lote 2 | Banco sige_dados | 5000 registros lidos
✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida
✅ Lote gravado/upsert: 5000 registros
📦 Lote 3 | Banco sige_dados | 5000 registros lidos
✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida
✅ Lote gravado/upsert: 5000 registros
📦 Lote 4 | Banco sige_dados | 5000 registros lidos
✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida
✅ Lote gravado/upsert: 5000 registros
📦 Lote 5 | Banco sige_dados | 5000 registros lidos
✅ Constraint UNIQUE garantida em "ID_UNICO" e PK antiga removida
✅ Lote gravado/upsert: 50